In [1]:
import cv2
import shutil
import torch
import torch.nn as nn
from torchvision import models, transforms
from ultralytics import YOLO

from pathlib import Path
from collections import Counter
from PIL import Image

In [2]:
# =========================================================
# CONFIG
# =========================================================

IMG_DIR = Path("../data/files/not_processed")
UNLABELED_DIR = Path("../data/unlabeled_crops/photos")

LABELED_DIR = Path("../data/labeled_dataset_final")

YOLO_MODEL_PATH = "../src/yolo11s.pt"
CLASSIFIER_MODEL_PATH = "../runs/final_model_benchmark/convnext_base/best_model.pth"

VEHICLE_CLASSES = [1, 2, 3, 5, 7]

CONF_THRESHOLD = 0.85
PADDING = 10

# If video files are present in `IMG_DIR`, sample every N_FRAMES
N_FRAMES = 90

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
# =========================================================
# CLASS MAPPING
# =========================================================

CLASS_NAMES = [
    'a_bikes',
    'b_moto',
    'c_pass',
    'd_light_comm',
    'e_heavy_rigid',
    'f_articulated',
    'g_bus',
    'h_agri'
]

KEY_MAP = {
    '1': 'a_bikes',
    '2': 'b_moto',
    '3': 'c_pass',
    '4': 'd_light_comm',
    '5': 'e_heavy_rigid',
    '6': 'f_articulated',
    '7': 'g_bus',
    '8': 'h_agri',
    '0': 'discard',
    ' ': 'discard'
}

# =========================================================
# CREATE DIRECTORIES
# =========================================================

for cls in CLASS_NAMES:
    (LABELED_DIR / cls).mkdir(parents=True, exist_ok=True)

(LABELED_DIR / "discard").mkdir(parents=True, exist_ok=True)
(LABELED_DIR / "skipped").mkdir(parents=True, exist_ok=True)

UNLABELED_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# LOAD YOLO
# =========================================================

print("Loading YOLO...")
yolo_model = YOLO(YOLO_MODEL_PATH)

# =========================================================
# LOAD CLASSIFIER
# =========================================================

print("Loading classifier...")

classifier = models.convnext_base(weights=None)
classifier.classifier[2] = nn.Linear(
    classifier.classifier[2].in_features,
    len(CLASS_NAMES)
)

classifier.load_state_dict(
    torch.load(CLASSIFIER_MODEL_PATH, map_location=DEVICE)
)

classifier.to(DEVICE)
classifier.eval()

# =========================================================
# TRANSFORMS
# =========================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# =========================================================
# STATS
# =========================================================

stats = Counter()

# =========================================================
# CLASSIFIER PREDICTION
# =========================================================

Loading YOLO...
Loading classifier...


In [5]:
@torch.no_grad()
def predict_crop(crop_bgr):

    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

    pil_img = Image.fromarray(crop_rgb)

    tensor = transform(pil_img).unsqueeze(0).to(DEVICE)

    outputs = classifier(tensor)

    probs = torch.softmax(outputs, dim=1)

    conf, pred = torch.max(probs, dim=1)

    pred_idx = pred.item()
    confidence = conf.item()

    return CLASS_NAMES[pred_idx], confidence

# =========================================================
# EXTRACT VEHICLE CROPS
# =========================================================

def extract_crops():

    # collect images and videos separately
    all_files = list(IMG_DIR.rglob("*"))

    image_files = [
        p for p in all_files
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    video_files = [
        p for p in all_files
        if p.suffix.lower() in [".mp4", ".avi", ".mov", ".mkv", ".wmv", ".flv"]
    ]

    print(f"Found {len(image_files)} source images and {len(video_files)} video files.")

    crop_counter = 0

    def _process_frame(frame, source_name):
        nonlocal crop_counter

        if frame is None:
            return

        results = yolo_model.predict(
            frame,
            classes=VEHICLE_CLASSES,
            conf=0.3,
            verbose=False
        )

        for result in results:

            boxes = result.boxes.xyxy.cpu().numpy()

            for idx, box in enumerate(boxes):

                x1, y1, x2, y2 = map(int, box)

                x1 = max(0, x1 - PADDING)
                y1 = max(0, y1 - PADDING)

                x2 = min(frame.shape[1], x2 + PADDING)
                y2 = min(frame.shape[0], y2 + PADDING)

                crop = frame[y1:y2, x1:x2]

                if crop.size == 0:
                    continue

                crop_name = f"{source_name}_crop_{idx}.jpg"

                crop_path = UNLABELED_DIR / crop_name

                cv2.imwrite(str(crop_path), crop)

                crop_counter += 1

    # process still images
    for img_path in image_files:

        frame = cv2.imread(str(img_path))

        if frame is None:
            continue

        _process_frame(frame, img_path.stem)

    # process videos by sampling every N_FRAMES
    for vid_path in video_files:

        cap = cv2.VideoCapture(str(vid_path))

        if not cap.isOpened():
            print(f"Failed to open video: {vid_path}")
            continue

        frame_idx = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % N_FRAMES == 0:
                source_name = f"{vid_path.stem}_frame_{frame_idx}"
                _process_frame(frame, source_name)

            frame_idx += 1

        cap.release()

    print(f"Extracted {crop_counter} crops.")

# =========================================================
# LABELING LOOP
# =========================================================

def labeling_loop():

    files = list(UNLABELED_DIR.glob("*.jpg"))

    print(f"Found {len(files)} unlabeled crops.")

    for i, img_path in enumerate(files):

        img = cv2.imread(str(img_path))

        if img is None:
            continue

        pred_label, conf = predict_crop(img)

        display = img.copy()

        display = cv2.resize(display, (600, 600))

        conf_text = f"{pred_label} ({conf:.2f})"

        if conf >= 0.98:
            color = (0, 255, 0)

        elif conf >= CONF_THRESHOLD:
            color = (0, 255, 255)

        else:
            color = (0, 0, 255)

        cv2.putText(
            display,
            f"[{i+1}/{len(files)}]",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255,255,255),
            2
        )

        cv2.putText(
            display,
            conf_text,
            (20, 90),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            color,
            2
        )

        y_offset = 140

        for cls_name, count in stats.items():

            cv2.putText(
                display,
                f"{cls_name}: {count}",
                (20, y_offset),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (200,200,200),
                2
            )

            y_offset += 30

        cv2.imshow(
            "Y=accept | N=manual | S=skip | SPACE/0=discard | ESC=exit",
            display
        )

        key = cv2.waitKey(0)

        if key == 27:
            break

        char = chr(key & 0xFF).lower()

        # =================================================
        # ACCEPT PREDICTION
        # =================================================

        if char == 'y':

            target_dir = LABELED_DIR / pred_label

            shutil.copy2(
                str(img_path),
                str(target_dir / img_path.name)
            )

            stats[pred_label] += 1

        # =================================================
        # MANUAL LABEL
        # =================================================

        elif char == 'n':

            print("\nManual class:")
            print("1=a_bikes")
            print("2=b_moto")
            print("3=c_pass")
            print("4=d_light_comm")
            print("5=e_heavy_rigid")
            print("6=f_articulated")
            print("7=g_bus")
            print("8=h_agri")
            print("0=discard")

            manual_key = input("Class: ").strip()

            if manual_key in KEY_MAP:

                label = KEY_MAP[manual_key]

                target_dir = LABELED_DIR / label

                shutil.copy2(
                    str(img_path),
                    str(target_dir / img_path.name)
                )

                stats[label] += 1

        # =================================================
        # SKIP
        # =================================================

        elif char == 's':

            shutil.copy2(
                str(img_path),
                str((LABELED_DIR / "skipped") / img_path.name)
            )

        # =================================================
        # DISCARD
        # =================================================

        elif char in ['0', ' ']:

            shutil.copy2(
                str(img_path),
                str((LABELED_DIR / "discard") / img_path.name)
            )

            stats["discard"] += 1

    cv2.destroyAllWindows()

In [6]:
# =========================================================
# RUN
# =========================================================

extract_crops()


print("\nDone.")

Found 553 source images and 2 video files.
Extracted 2997 crops.

Done.


In [7]:
labeling_loop()


Found 2997 unlabeled crops.


QFontDatabase: Cannot find font directory /home/haarmeggido/Repos/heimdall-counter-raw/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/haarmeggido/Repos/heimdall-counter-raw/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/haarmeggido/Repos/heimdall-counter-raw/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/haarmeggido/Repos/heimdall-counter-raw/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io


Manual class:
1=a_bikes
2=b_moto
3=c_pass
4=d_light_comm
5=e_heavy_rigid
6=f_articulated
7=g_bus
8=h_agri
0=discard

Manual class:
1=a_bikes
2=b_moto
3=c_pass
4=d_light_comm
5=e_heavy_rigid
6=f_articulated
7=g_bus
8=h_agri
0=discard

Manual class:
1=a_bikes
2=b_moto
3=c_pass
4=d_light_comm
5=e_heavy_rigid
6=f_articulated
7=g_bus
8=h_agri
0=discard

Manual class:
1=a_bikes
2=b_moto
3=c_pass
4=d_light_comm
5=e_heavy_rigid
6=f_articulated
7=g_bus
8=h_agri
0=discard

Manual class:
1=a_bikes
2=b_moto
3=c_pass
4=d_light_comm
5=e_heavy_rigid
6=f_articulated
7=g_bus
8=h_agri
0=discard

Manual class:
1=a_bikes
2=b_moto
3=c_pass
4=d_light_comm
5=e_heavy_rigid
6=f_articulated
7=g_bus
8=h_agri
0=discard

Manual class:
1=a_bikes
2=b_moto
3=c_pass
4=d_light_comm
5=e_heavy_rigid
6=f_articulated
7=g_bus
8=h_agri
0=discard

Manual class:
1=a_bikes
2=b_moto
3=c_pass
4=d_light_comm
5=e_heavy_rigid
6=f_articulated
7=g_bus
8=h_agri
0=discard

Manual class:
1=a_bikes
2=b_moto
3=c_pass
4=d_light_comm
5=e_he